In [ ]:
import matplotlib.pyplot as plt

labels = ["Clean", "FGSM", "PGD", "Авторская атака"]
acc = [0.9640, 0.1220, 0.0400, 0.1670]

plt.figure(figsize=(8,5))
bars = plt.bar(labels, acc)
plt.ylim(0, 1.05)
plt.title("Accuracy модели до и после атак")
plt.ylabel("Accuracy")
plt.grid(axis="y", linestyle="--", alpha=0.5)

for b in bars:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
             f"{b.get_height():.3f}", ha="center")

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

labels = ["Clean", "FGSM", "PGD", "Авторская атака"]
f1 = [0.9640, 0.0986, 0.0210, 0.1159]

plt.figure(figsize=(8,5))
bars = plt.bar(labels, f1)
plt.ylim(0, 1.05)
plt.title("F1-score модели до и после атак")
plt.ylabel("F1-score")
plt.grid(axis="y", linestyle="--", alpha=0.5)

for b in bars:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
             f"{b.get_height():.3f}", ha="center")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

# ПУТЬ К ЧЕКПОИНТУ:
MODEL_PATH = r"melanoma_attacks/checkpoint-12670"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Грузим процессор и модель:
processor = AutoImageProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

# НАСТРОЙКИ ИЗОБРАЖЕНИЙ:
original_images = [
    "balanced_Image/MEL/ISIC_0000298.jpg",
    "balanced_Image/NV/ISIC_0000012.jpg",
    "balanced_Image/BCC/ISIC_0025102.jpg",
    "balanced_Image/SCC/ISIC_0027506.jpg"
]
attacked_images = [
    "isic_1000_attacked/MEL/ISIC_0000298.jpg",
    "isic_1000_attacked/NV/ISIC_0000012.jpg",
    "isic_1000_attacked/BCC/ISIC_0025102.jpg",
    "isic_1000_attacked/SCC/ISIC_0027506.jpg"
]

# ФУНКЦИЯ ПРЕДСКАЗАНИЯ:
@torch.no_grad()
def predict_label_and_prob(img: Image.Image):
    inputs = processor(images=img, return_tensors="pt").to(device)
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred_id = torch.argmax(probs).item()
    pred_label = model.config.id2label[pred_id]
    pred_prob = probs[pred_id].item()
    return pred_label, pred_prob

#ВИЗУАЛИЗАЦИЯ:
plt.rcParams.update({
    'font.size': 11,
    'font.family': 'Arial'
})
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    # Оригиналы (RGB):
    orig_img = Image.open(original_images[i]).convert("RGB")
    orig_label, orig_prob = predict_label_and_prob(orig_img)
    axes[0, i].imshow(orig_img)
    axes[0, i].axis("off")
    axes[0, i].set_title(
        f"Оригинал {i+1}\n"
        f"{orig_label} ({orig_prob:.3f})",
        fontsize=11
    )
    # Атакованные (RGB):
    atk_img = Image.open(attacked_images[i]).convert("RGB")
    atk_label, atk_prob = predict_label_and_prob(atk_img)

    axes[1, i].imshow(atk_img)
    axes[1, i].axis("off")
    axes[1, i].set_title(
        f"Атака {i+1}\n"
        f"{atk_label} ({atk_prob:.3f})",
        fontsize=11
    )
fig.suptitle(
    "Сравнение исходных и атакованных изображений (предсказание модели)\n",
    fontsize=12, weight='bold'
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
fig.subplots_adjust(hspace=0.45)
plt.show()


In [ ]:
import os
import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
import pandas as pd

# Загрузка изображения:
def load_image(path, size=(224, 224)):
    img = Image.open(path).convert("RGB")
    img = img.resize(size)
    return np.array(img) / 255.0

# Метрики сходства:
def mse(imageA, imageB):
    return np.mean((imageA - imageB) ** 2)

def psnr(imageA, imageB):
    mse_val = mse(imageA, imageB)
    if mse_val == 0:
        return float("inf")
    return 20 * np.log10(1.0 / np.sqrt(mse_val))

# Папки:
orig_folder = "balanced_Image"
adv_folder = "isic_1000_attacked"
results = []

# Обход папок и вычисление метрик:
for root, _, files in os.walk(orig_folder):
    for file in files:
        if file.lower().endswith((".jpg", ".png", ".jpeg")):
            orig_path = os.path.join(root, file)
            rel_path = os.path.relpath(orig_path, orig_folder)
            adv_path = os.path.join(adv_folder, rel_path)
            if not os.path.exists(adv_path):
                continue
            orig = load_image(orig_path)
            adv = load_image(adv_path)
            mse_val = mse(orig, adv)
            psnr_val = psnr(orig, adv)
            ssim_val = ssim(orig, adv, channel_axis=-1, data_range=1.0)
            results.append((rel_path, mse_val, psnr_val, ssim_val))

# Создание DataFrame:
df = pd.DataFrame(results, columns=["Файл", "MSE", "PSNR", "SSIM"])

# Подсчёт средних значений:
mean_values = df[["MSE", "PSNR", "SSIM"]].mean()
print("\nСредние значения по всем изображениям:")
print(mean_values)

# Визуализация:
plt.rcParams.update({'font.size': 11, 'font.family': 'Arial'})
df[["MSE", "PSNR", "SSIM"]].hist(
    bins=30,
    figsize=(12, 5),
    edgecolor="black"
)

# Заголовки и подписи:
plt.suptitle("Распределение метрик сходства\n", fontsize=13, weight="bold")

axes = plt.gcf().axes
labels = [
    ("MSE (среднеквадратичная ошибка)"),
    ("PSNR (пиковое отношение сигнал/шум)"),
    ("SSIM (структурное сходство)")
]
for ax, (ru) in zip(axes, labels):
    ax.set_xlabel(f"{ru}", fontsize=10)
    ax.set_ylabel("Количество изображений", fontsize=10)
    ax.grid(alpha=0.4, linestyle="--")
plt.tight_layout(rect=[0, 0, 1, 0.95])

plt.show()


In [ ]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity

# Папки:
orig_dir = "balanced_Image/AK"
adv_dir = "isic_1000_attacked/AK"

# Выбор 3х случайных изображений:
files = [f for f in os.listdir(orig_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
sample_files = random.sample(files, 3)

plt.rcParams.update({'font.size': 12, 'font.family': 'Arial'})
fig, axes = plt.subplots(len(sample_files), 3, figsize=(12, 9))

for row_idx, filename in enumerate(sample_files):
    orig_path = os.path.join(orig_dir, filename)
    adv_path = os.path.join(adv_dir, filename)
    orig_img = mpimg.imread(orig_path)
    adv_img = mpimg.imread(adv_path)
    # Корректный диапазон данных для PSNR/SSIM:
    # если float и максимум <= 1.0 -> диапазон [0,1]
    # иначе считаем как [0,255]
    data_range = 1.0 if orig_img.max() <= 1.0 else 255
    # Метрики:
    mse_val = mean_squared_error(orig_img, adv_img)
    psnr_val = peak_signal_noise_ratio(orig_img, adv_img, data_range=data_range)
    ssim_val = structural_similarity(orig_img, adv_img, channel_axis=-1, data_range=data_range)
    # Оригинал:
    axes[row_idx, 0].imshow(orig_img)
    axes[row_idx, 0].axis("off")
    if row_idx == 0:
        axes[row_idx, 0].set_title("Оригинал", fontsize=14, pad=25)
    # Атакованное:
    axes[row_idx, 1].imshow(adv_img)
    axes[row_idx, 1].axis("off")
    if row_idx == 0:
        axes[row_idx, 1].set_title("Атакованное", fontsize=14, pad=25)
    # Метрики:
    axes[row_idx, 2].text(
        0.5, 0.5,
        f"MSE:  {mse_val:.4f}\n"
        f"PSNR: {psnr_val:.2f}\n"
        f"SSIM: {ssim_val:.3f}",
        fontsize=13,
        va="center",
        ha="center",
        weight="bold"
    )
    axes[row_idx, 2].axis("off")
    if row_idx == 0:
        axes[row_idx, 2].set_title("Метрики оценивания", fontsize=14, pad=25)
# Общий заголовок:
plt.subplots_adjust(top=0.88)
fig.suptitle(
    "Сравнение оригинальных и атакованных изображений\n",
    fontsize=15, weight="bold", y=0.98
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()
